In [ ]:
# Importing everything we need

import os, json, pickle, random, time, re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
# Configuration of model run(s) and save it so we know which run yields which results
CONFIG = {
    "experiment_name": "slp_tfidf_extremism_v1",
    "random_seed": 30,

    "data_path": "/kaggle/input/datasets/anthony73/kaggle-environment-extremism-detection/extremism_dataset_final.csv",

    # Change these after checking df.columns
    "text_col": "Original_Message",
    "label_col": "Extremism_Label",

    "test_size": 0.2,
    "max_features": 20000,
    "ngram_range": (1, 2),
    "min_df": 2,
    "max_df": 0.95,

    "batch_size": 32,
    "epochs": 30,
    "lr": 0.01,
    "weight_decay": 0.0,
    "threshold": 0.5,

    # SHAP settings
    "shap_background_size": 100,
    "shap_explain_size": 200,
}

OUTPUT_DIR = Path("/kaggle/working") / CONFIG["experiment_name"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(CONFIG["random_seed"])

In [ ]:
# Loading and inspecting our dataset
df = pd.read_csv(CONFIG["data_path"])

print(df.shape)
display(df.head())
print(df.columns)
print(df[CONFIG["label_col"]].value_counts(dropna=False))

In [ ]:
# Standard data cleanup protocol and pre-processing

def clean_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df = df[[CONFIG["text_col"], CONFIG["label_col"]]].dropna()
df[CONFIG["text_col"]] = df[CONFIG["text_col"]].apply(clean_text)

# Ensure binary integer labels (EXTREMIST = 1, NON_EXTREMIST = 0)
# Only map if the column contains strings (to avoid re-run errors)
if df[CONFIG["label_col"]].dtype == object:
    label_conv_map = {"NON_EXTREMIST": 0, "EXTREMIST": 1}
    df[CONFIG["label_col"]] = df[CONFIG["label_col"]].map(label_conv_map)

if df[CONFIG["label_col"]].isna().any():
    raise ValueError("Unexpected label values found.")

X = df[CONFIG["text_col"]].values
y = df[CONFIG["label_col"]].values

print("Dataset size:", len(df))
print("Positive rate:", y.mean())

In [ ]:
# Inspecting the data after processing
print(df.shape)
display(df.head())
print(df.columns)
print(df[CONFIG["label_col"]].value_counts(dropna=False))

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X,
    y,
    test_size=CONFIG["test_size"],
    random_state=CONFIG["random_seed"],
    stratify=y
)

print(len(X_train_text), len(X_test_text))

# Ensure stratification is functioning
print("Train positive rate:", y_train.mean())
print("Test positive rate:", y_test.mean())

In [ ]:
# Vectorizing (TF-IDF) using the configuration settings
vectorizer = TfidfVectorizer(
    max_features=CONFIG["max_features"],
    ngram_range=CONFIG["ngram_range"],
    min_df=CONFIG["min_df"],
    max_df=CONFIG["max_df"],
    lowercase=True,
    stop_words="english"
)

X_train_vec = vectorizer.fit_transform(X_train_text)
X_test_vec = vectorizer.transform(X_test_text)

print(X_train_vec.shape)
print(X_test_vec.shape)

In [ ]:
# Conversion to PyTorch tensors
X_train_tensor = torch.tensor(X_train_vec.toarray(), dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_vec.toarray(), dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.reshape(-1, 1), dtype=torch.float32)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True
)

In [ ]:
# Defining our SLP model
class SLPTextClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        logits = self.linear(x)
        return logits

input_dim = X_train_tensor.shape[1]
model = SLPTextClassifier(input_dim)
print(model)

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(
    model.parameters(),
    lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"]
)

history = []

best_f1 = 0
for epoch in range(CONFIG["epochs"]):
    model.train()
    total_loss = 0

    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        logits = model(batch_X)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    model.eval()
    with torch.no_grad():
        test_logits = model(X_test_tensor)
        test_probs = torch.sigmoid(test_logits).numpy().ravel()
        test_preds = (test_probs >= CONFIG["threshold"]).astype(int)

    acc = accuracy_score(y_test, test_preds)
    f1 = f1_score(y_test, test_preds, zero_division=0)
    precision = precision_score(y_test, test_preds, zero_division=0)
    recall = recall_score(y_test, test_preds, zero_division=0)

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), OUTPUT_DIR / "slp_model_best.pt")

    history.append({
        "epoch": epoch + 1,
        "train_loss": avg_loss,
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
    })

    print(
        f"Epoch {epoch+1:03d} | "
        f"Loss: {avg_loss:.4f} | "
        f"Acc: {acc:.4f} | "
        f"F1: {f1:.4f} | "
        f"Precision: {precision:.4f} | "
        f"Recall: {recall:.4f}"
    )

history_df = pd.DataFrame(history)
display(history_df.tail())

In [ ]:
plt.figure()
plt.plot(history_df["epoch"], history_df["train_loss"])
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss")
plt.show()

plt.figure()
plt.plot(history_df["epoch"], history_df["f1"])
plt.xlabel("Epoch")
plt.ylabel("F1 Score")
plt.title("Test F1 Over Epochs")
plt.show()

In [ ]:
model.eval()

with torch.no_grad():
    test_logits = model(X_test_tensor)
    test_probs = torch.sigmoid(test_logits).numpy().ravel()
    test_preds = (test_probs >= CONFIG["threshold"]).astype(int)

print(classification_report(y_test, test_preds, zero_division=0))

cm = confusion_matrix(y_test, test_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title("Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import roc_auc_score, confusion_matrix

tn, fp, fn, tp = confusion_matrix(y_test, test_preds).ravel()
metrics = {
    "accuracy": accuracy_score(y_test, test_preds),
    "f1": f1_score(y_test, test_preds, zero_division=0),
    "precision": precision_score(y_test, test_preds, zero_division=0),
    "recall": recall_score(y_test, test_preds, zero_division=0),
    "roc_auc": roc_auc_score(y_test, test_probs),  
    "true_positives": int(tp),
    "true_negatives": int(tn),
    "false_positives": int(fp),
    "false_negatives": int(fn),  
}

import datetime

metrics["experiment_name"] = CONFIG["experiment_name"]
metrics["timestamp"] = datetime.datetime.now().isoformat()
metrics["dataset_size"] = len(df)
metrics["train_size"] = len(X_train_text)
metrics["test_size"] = len(X_test_text)
metrics["positive_rate"] = float(y.mean())
metrics["epochs_trained"] = CONFIG["epochs"]
metrics["best_epoch"] = history_df.loc[history_df["f1"].idxmax(), "epoch"]

with open(OUTPUT_DIR / "tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

with open(OUTPUT_DIR / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=4, default=str)

history_df.to_csv(OUTPUT_DIR / "training_history.csv", index=False)

metrics = {
    "accuracy": accuracy_score(y_test, test_preds),
    "f1": f1_score(y_test, test_preds, zero_division=0),
    "precision": precision_score(y_test, test_preds, zero_division=0),
    "recall": recall_score(y_test, test_preds, zero_division=0),
}

with open(OUTPUT_DIR / "final_metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

print("Saved to:", OUTPUT_DIR)
print(metrics)

In [ ]:
def run_experiment(config):
    set_seed(config["random_seed"])

    # Vectorize
    vectorizer = TfidfVectorizer(
        max_features=config["max_features"], ngram_range=config["ngram_range"],
        min_df=config["min_df"], max_df=config["max_df"],
        lowercase=True, stop_words="english"
    )
    X_train_vec = vectorizer.fit_transform(X_train_text)
    X_test_vec = vectorizer.transform(X_test_text)

    # Tensors
    X_train_tensor = torch.tensor(X_train_vec.toarray(), dtype=torch.float32)
    X_test_tensor = torch.tensor(X_test_vec.toarray(), dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)
    train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor),
                              batch_size=config["batch_size"], shuffle=True)

    # Model
    model = SLPTextClassifier(X_train_tensor.shape[1])
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])

    # Train
    best_f1 = 0
    for epoch in range(config["epochs"]):
        model.train()
        total_loss = 0
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(batch_X), batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        model.eval()
        with torch.no_grad():
            test_probs = torch.sigmoid(model(X_test_tensor)).numpy().ravel()
            test_preds = (test_probs >= config["threshold"]).astype(int)
        f1 = f1_score(y_test, test_preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_preds = test_preds  # save preds at best epoch
            best_probs = test_probs

    # Final metrics
    return {
        "experiment_name": config["experiment_name"],
        "lr": config["lr"],
        "max_features": config["max_features"],
        "ngram_range": str(config["ngram_range"]),
        "weight_decay": config["weight_decay"],
        "batch_size": config["batch_size"],
        "best_f1": best_f1,
        "accuracy": accuracy_score(y_test, test_preds),
        "precision": precision_score(y_test, test_preds, zero_division=0),
        "recall": recall_score(y_test, test_preds, zero_division=0),
    }

In [ ]:
import itertools

ablation_grid = {
    "lr":           [0.001, 0.0005, 0.0001],   # lower — 0.01 is clearly too high
    "weight_decay": [1e-3, 1e-4, 1e-5],         # add regularization, 0.0 is clearly too low
    "max_features": [10000, 20000, 50000],       # keep this range, reasonable
    "ngram_range":  [(1, 1), (1, 2), (1, 3)],   # keep, worth exploring
    "batch_size":   [32, 64],                    # drop 128, likely too large for your dataset
}

# Generate all combinations
keys = list(ablation_grid.keys())
combinations = list(itertools.product(*ablation_grid.values()))
print(f"Total experiments: {len(combinations)}")

In [ ]:
all_results = []

for i, combo in enumerate(combinations):
    config = {**CONFIG}  # start from your base config
    for k, v in zip(keys, combo):
        config[k] = v
    config["experiment_name"] = f"ablation_{i:04d}"
    
    print(f"\n[{i+1}/{len(combinations)}] Running: { {k: v for k, v in zip(keys, combo)} }")
    result = run_experiment(config)
    all_results.append(result)

results_df = pd.DataFrame(all_results).sort_values("best_f1", ascending=False)
results_df.to_csv("/kaggle/working/ablation_results.csv", index=False)
display(results_df.head(10))